# einsum-contraction — ex2: lambertian dot product via index contraction

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `einsum-contraction`. Running the final beacon cell reports progress against the `Einsum: Index contraction semantics` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Einsum: Index contraction semantics` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`einsum-contraction`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "einsum-contraction"
DD_SUBTOPIC = "Einsum: Index contraction semantics"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Einsum index rules — quick refresher

`einsum` is governed by **two simple index rules**:

1. **An index that appears on BOTH sides of `->`** is preserved (carried through unchanged on every operand and the output). Think of it as a broadcasted/batch axis.
2. **An index that appears on the INPUT but NOT on the output** is *summed-contracted*: einsum multiplies aligned entries and sums them away.

**Repeated index on the same operand.** If an index appears twice on one operand (e.g. `'i i -> i'`), it pulls the **diagonal**. If `'i i ->'` with nothing on the right, it pulls the diagonal *and* sums it — that's the trace.

**Worked examples of the rules in action:**
- `'i j, j k -> i k'`: `j` repeats across operands → sum-contracted (matmul).
- `'i j -> i'`: `j` dropped from rhs → row sum.
- `'i j -> '`: all indices dropped → grand sum (scalar).
- `'i j, i j -> i j'`: nothing dropped → elementwise product (Hadamard).
- `'i, j -> i j'`: nothing repeated, both kept → outer product.

**Why this matters.** Once you internalise these two rules, you can read *any* einsum pattern from left to right and predict the output without running it. That's the whole pedagogical payoff.

### Exercise 2 — lambertian dot product via index contraction

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the einsum index rules to compute a batched dot product between per-triangle surface normals and a single light direction, using a pattern where one index is shared (and contracted) and one is batch (and preserved).
> Keywords: dot-product, lambertian, ray-tracing, lighting
> ```

**KCs targeted:** `einsum-repeated-index-sums`, `einsum-batch-axis-passthrough`

Implement `ex2_lambertian_intensity(normals, light)` — the dot product that drives ARENA's Lambertian shading bonus exercise:

- `normals` has shape `(NT, 3)` — one unit-length normal per triangle.
- `light` has shape `(3,)` — a single light direction.
- Return `(NT,)` — the dot product of each normal with the light.

**Constraint.** You must use `einops.einsum` (not `@`, not `(normals * light).sum(-1)`). The whole point of the drill is naming the indices.

**Build the pattern from the rules.** You need `nt` to survive (every triangle gets its own output) and `dims` to be summed (that's the dot product). Pattern: `'nt dims, dims -> nt'` — `dims` is shared across operands and missing from rhs → contracted. `nt` is on the first operand only and on rhs → preserved.

Don't clamp the output here (the ARENA Lambertian step does `max(0, ...)` separately to suppress backward-facing normals).

In [ ]:
def ex2_lambertian_intensity(normals: Tensor, light: Tensor) -> Tensor:
    return einops.einsum(normals, light, 'nt dims, dims -> nt')


<details><summary>Solution</summary>

```python
def ex2_lambertian_intensity(normals: Tensor, light: Tensor) -> Tensor:
    return einops.einsum(normals, light, 'nt dims, dims -> nt')
```

**Reading the pattern through the rules.** `'nt dims, dims -> nt'`:
- `dims` appears on both operands but NOT on rhs → contracted (summed). That's the dot product.
- `nt` appears on the first operand and on rhs → preserved. Each triangle gets its own scalar.
- `light` has no `nt` axis but lives in the same `dims` space — einsum broadcasts it against every triangle. No `repeat` needed.

**Why we don't write `nt dims, nt dims -> nt`.** You *could* first `repeat` the light to `(NT, 3)` and then contract — that's the explicit form. But einsum is happy to broadcast missing batch axes for free. The shorter pattern is idiomatic.

**Where ARENA uses this exactly.** `einops.einsum(normals, light.to(device), 'nt dims, dims -> nt')` is the Lambertian step in `raytrace_mesh_lighting` (0_1_9). The downstream `t.where(intensity > 0, intensity, 0)` is the back-face clip — distinct atom, not this one.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()